In [2]:
import serial
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

class ImagePacketDecoder:
    """解析固定大小图像包，每包第一个字节为 0xFF，剩余为 u8 像素数据。"""
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.packet_size = width * height + 1
        self.reset()

    def reset(self):
        self.state = "WAIT_HEADER"
        self.buffer = np.zeros(self.width * self.height, dtype=np.uint8)
        self.addr = 0

    def feed_byte(self, byte):
        if self.state == "WAIT_HEADER":
            if byte == 0xFF:
                self.state = "RECEIVE_DATA"
            return None
        elif self.state == "RECEIVE_DATA":
            self.buffer[self.addr] = byte
            self.addr += 1
            if self.addr >= self.buffer.size:
                img = self.buffer.reshape((self.height, self.width))
                self.reset()
                return img
        return None

def run_uart_with_fps(port="COM8", baud=115200, width=160, height=120):
    decoder = ImagePacketDecoder(width, height)
    
    frame_count = 0
    last_time = time.time()
    
    with serial.Serial(port, baud, timeout=0.01) as ser:
        print("等待圖像幀...")
        
        while True:
            raw = ser.read(1)
            if not raw:
                continue
            
            img = decoder.feed_byte(raw[0])
            if img is not None:
                frame_count += 1
                now = time.time()
                dt = now - last_time
                fps = 1.0 / dt if dt > 0 else 0
                last_time = now
                
                clear_output(wait=True)
                plt.imshow(img, cmap="gray")
                plt.title(f"Frame #{frame_count} | FPS: {fps:.2f}")
                plt.axis('off')
                plt.show()

run_uart_with_fps("COM4", 5000000, 160, 120)

KeyboardInterrupt: 